# Experiment Setup

**(LRHS log 010)**

Definition of the numerical test setup

## Table of Contents

## Defining water depth and wave conditions [2026-01-24]

- In the Agarwal 2022 paper, we took quite high wave freqs $\omega >1$
- Here as well, for us wave making only worked well for $\omega>1.5$
- The following is the plot of wave amplitude in the middle of the domain for the range of freqs.


Configuration `depth=10m`
![alt text](log_v010/mem_dxPrb_orig_H10.png)

- This wave generation doesnt work well with typical wave spectra.
- I need to tune the $mu_0$ in wave making sponge layer to make it work for low frequencies for this water depth.
- **AIM** Therefore the aim is to make it work in a deeper water depth instead.

Typically for a regular sea-state `Hs = 3m, Tp = 10.0s` the JONSWAP spectrum is

- **Spectral Density** <br>
  ![alt text](log_v010/jonswap_specdensity_Hs3_Tp10p0.png) 
- **Amplitude Spec** <br> 
  ![alt text](log_v010/jonswap_amplitude_Hs3_Tp10p0.png) 
 


Typically for a extreme sea-state `Hs = 7m, Tp = 12.4s` the JONSWAP spectrum is

- **Spectral Density**<br>
  ![alt text](log_v010/jonswap_specdensity_Hs7_Tp12p4.png) 
- **Amplitude Spec**<br>
  ![alt text](log_v010/jonswap_amplitude_Hs7_Tp12p4.png) 





## Radiation BC on inlet [2026-01-24]


I have been trying in various way to implement a radiation BC at the inlet as well.

Typicall we have in te weak form

```Julia
a((ϕ,κ,η),(w,u,v)) =      
      ∫(  ∇(w)⋅∇(ϕ) )dΩ   +
      ∫( -w * ∇ϕ⋅n )dΓot 
```

which we make as 

```Julia
a((ϕ,κ,η),(w,u,v)) =      
      ∫(  ∇(w)⋅∇(ϕ) )dΩ   +
      ∫( -w * im * k * ϕ )dΓot
```

given the Radiation BC

```Julia
∂ϕ/∂t + c∇ϕ⋅n = 0
∇ϕ⋅n = -1/c ∂ϕ/∂t = im*k*ϕ
```

Finally I tried the Radiation BC with Input in the following form.

![alt text](log_v010/Wellens_2020_snap1.png) <br>
Reference: Wellens (2020) Computers and Fluids

- This is the version that I have tried to implement a number of times before.
- This time I paid closer attention to the expressions in the implementation and i realised a **MISTAKE**

**Wrong Implementation**
```Julia
  ηᵢₙ(x) = η₀*exp(im*k*x[1] + im*α)
  ϕᵢₙ(x) = -im*(η₀*ω/k)*(cosh(k*(H0 + x[2])) / 
    sinh(k*H0))*exp(im*k*x[1] + im*α)
  vxᵢₙ(x) = (η₀*ω)*(cosh(k*(H0 + x[2])) / #MISTAKE 
    sinh(k*H0))*exp(im*k*x[1] + im*α) 
  vzfsᵢₙ(x) = -im*ω*η₀*exp(im*k*x[1] + im*α) #???

  a((ϕ,κ),(w,u)) += ∫( -w * im * k * ϕ )dΓot + ∫( -w * im * k * ϕ )dΓin 
  l((w,u)) =  ∫( w*vxᵢₙ )dΓin - ∫( w * im * k * ϕᵢₙ )dΓin
```

Due to this mistake, when i implementation the wavemaking + Radiation BC, I got zero input wave.

**Correct Implementation**
```Julia
  ηᵢₙ(x) = η₀*exp(im*k*x[1] + im*α)
  ϕᵢₙ(x) = -im*(η₀*ω/k)*(cosh(k*(H0 + x[2])) / 
    sinh(k*H0))*exp(im*k*x[1] + im*α)
  
  # vxᵢₙ = (∇ϕᵢₙ ⋅ nᵢₙ) = ∂ϕᵢₙ/∂x * -1
  vxᵢₙ(x) = -(η₀*ω)*(cosh(k*(H0 + x[2])) / 
    sinh(k*H0))*exp(im*k*x[1] + im*α)

  a((ϕ,κ),(w,u)) += ∫( -w * im * k * ϕ )dΓot + ∫( -w * im * k * ϕ )dΓin 
  l((w,u)) =  ∫( w*vxᵢₙ )dΓin - ∫( w * im * k * ϕᵢₙ )dΓin
```

Now everything works.

Henceforth, to avoid confusion, I instead write in the original weak form using $\nabla \phi\text{in}$, instead of $v\text{in}$

```Julia
l((w,u)) =  ∫( w*(∇ϕin⋅nΓin) )dΓin - ∫( w * im * k * ϕᵢₙ )dΓin
```




Difference is

**Sponge layer wavemaking**
![alt text](log_v010/mem_dxPrb_orig_H10.png)

**Correct Radiation BC Implementation**
![alt text](log_v010/mem_dxPrb_orig_H10_Corrected.png)